# CSD622 — Pattern Recognition & Data Mining
## Python Implementations — All Algorithms
**Tutor: Dr A.J Fofanah** | Master of Science Programme

Each section implements one algorithm end-to-end with a real dataset.


## 0. Shared Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import (
    load_iris, load_breast_cancer, load_wine,
    fetch_20newsgroups, load_digits, make_moons,
    make_classification, fetch_california_housing
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, average_precision_score
)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

---
## 1. Naive Bayes Classifier
**Dataset:** 20 Newsgroups (text classification, 20 categories, ~18,000 documents)  
**Algorithm:** Multinomial Naive Bayes with TF-IDF features and Laplace smoothing


In [ ]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

categories = ['rec.sport.hockey', 'sci.med', 'comp.graphics',
              'talk.politics.guns', 'rec.autos']

news_train = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers','footers','quotes'))
news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers','footers','quotes'))

pipe_mnb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, sublinear_tf=True)),
    ('clf',   MultinomialNB(alpha=1.0))
])
pipe_mnb.fit(news_train.data, news_train.target)
y_pred = pipe_mnb.predict(news_test.data)

print("Accuracy:", accuracy_score(news_test.target, y_pred))
print(classification_report(news_test.target, y_pred,
      target_names=news_test.target_names))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    news_test.target, y_pred,
    display_labels=[c.split('.')[-1] for c in categories],
    ax=ax, cmap='Blues')
ax.set_title('Multinomial Naive Bayes — 20 Newsgroups')
plt.tight_layout()
plt.show()

In [ ]:
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, random_state=42, stratify=data.target)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
print("Gaussian NB (Iris) accuracy:", gnb.score(X_test, y_test))
print("Per-class means (μ) per feature:")
print(pd.DataFrame(gnb.theta_, index=data.target_names,
                   columns=data.feature_names).round(3))

---
## 2. k-Nearest Neighbours (k-NN)
**Dataset:** Breast Cancer Wisconsin (569 samples, 30 features, binary: malignant/benign)  
**Task:** Classification with distance metric comparison and optimal k selection


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

bc = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    bc.data, bc.target, test_size=0.2, random_state=42, stratify=bc.target)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

k_range = range(1, 31)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
             X_tr_s, y_tr, cv=5, scoring='f1').mean() for k in k_range]

best_k = k_range[np.argmax(cv_scores)]
print(f"Optimal k = {best_k}  (CV F1 = {max(cv_scores):.4f})")

knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_tr_s, y_tr)
y_pred = knn.predict(X_te_s)
print(classification_report(y_te, y_pred, target_names=bc.target_names))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(k_range, cv_scores, 'o-', color='steelblue')
axes[0].axvline(best_k, color='red', linestyle='--', label=f'k={best_k}')
axes[0].set(xlabel='k', ylabel='CV F1', title='k-NN: Optimal k Selection')
axes[0].legend()

cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=bc.target_names, yticklabels=bc.target_names)
axes[1].set_title(f'k-NN Confusion Matrix (k={best_k})')
plt.tight_layout(); plt.show()

In [ ]:
metrics = ['euclidean', 'manhattan', 'chebyshev']
for m in metrics:
    acc = cross_val_score(KNeighborsClassifier(n_neighbors=best_k, metric=m),
                          X_tr_s, y_tr, cv=5).mean()
    print(f"{m:12s}  CV Accuracy = {acc:.4f}")

---
## 3. Decision Trees — CART
**Dataset:** Wine Recognition (178 samples, 13 chemical features, 3 cultivars)  
**Task:** Classification with information gain, pruning, and feature importance


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

wine = load_wine()
X_tr, X_te, y_tr, y_te = train_test_split(
    wine.data, wine.target, test_size=0.25, random_state=42, stratify=wine.target)

dt_full = DecisionTreeClassifier(criterion='gini', random_state=42)
dt_full.fit(X_tr, y_tr)
print(f"Full tree — depth {dt_full.get_depth()}, "
      f"leaves {dt_full.get_n_leaves()}, "
      f"test acc {dt_full.score(X_te, y_te):.4f}")

path = dt_full.cost_complexity_pruning_path(X_tr, y_tr)
alphas = path.ccp_alphas[:-1]

cv_scores = [cross_val_score(
    DecisionTreeClassifier(ccp_alpha=a, random_state=42),
    X_tr, y_tr, cv=5).mean() for a in alphas]

best_alpha = alphas[np.argmax(cv_scores)]
dt_pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
dt_pruned.fit(X_tr, y_tr)
print(f"Pruned tree  — depth {dt_pruned.get_depth()}, "
      f"leaves {dt_pruned.get_n_leaves()}, "
      f"test acc {dt_pruned.score(X_te, y_te):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(alphas, cv_scores, 'o-', color='seagreen')
axes[0].axvline(best_alpha, color='red', linestyle='--',
                label=f'α={best_alpha:.4f}')
axes[0].set(xlabel='alpha', ylabel='CV Accuracy', title='Cost-Complexity Pruning')
axes[0].legend()

plot_tree(dt_pruned, feature_names=wine.feature_names,
          class_names=wine.target_names, filled=True, ax=axes[1])
axes[1].set_title('Pruned Decision Tree')

fi = pd.Series(dt_pruned.feature_importances_,
               index=wine.feature_names).sort_values(ascending=True)
fi.plot.barh(ax=axes[2], color='steelblue')
axes[2].set_title('Feature Importances (Gini)')
plt.tight_layout(); plt.show()

In [ ]:
print(export_text(dt_pruned, feature_names=list(wine.feature_names)))

---
## 4. Support Vector Machine (SVM)
**Dataset:** Breast Cancer Wisconsin — full 30-feature dataset  
**Task:** Binary classification with RBF kernel, joint C/γ grid search, decision boundary visualisation


In [ ]:
from sklearn.svm import SVC

bc = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    bc.data, bc.target, test_size=0.2, random_state=42, stratify=bc.target)

from sklearn.pipeline import Pipeline
pipe = Pipeline([('sc', StandardScaler()), ('svm', SVC(probability=True))])

param_grid = {'svm__C': [0.1, 1, 10, 100],
              'svm__gamma': ['scale', 0.01, 0.1],
              'svm__kernel': ['rbf', 'linear']}

gs = GridSearchCV(pipe, param_grid,
                  cv=StratifiedKFold(5), scoring='f1', n_jobs=-1)
gs.fit(X_tr, y_tr)

print("Best params:", gs.best_params_)
print("Best CV F1: ", gs.best_score_)

y_pred  = gs.predict(X_te)
y_proba = gs.predict_proba(X_te)[:, 1]
print(classification_report(y_te, y_pred, target_names=bc.target_names))
print(f"AUC-ROC: {roc_auc_score(y_te, y_proba):.4f}")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
scaler2 = StandardScaler()
X_2d = pca.fit_transform(scaler2.fit_transform(bc.data))
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_2d, bc.target, test_size=0.2, random_state=42, stratify=bc.target)

svm2d = SVC(kernel='rbf', C=10, gamma=0.1)
svm2d.fit(X_tr2, y_tr2)

xx, yy = np.meshgrid(np.linspace(X_2d[:,0].min()-1, X_2d[:,0].max()+1, 300),
                      np.linspace(X_2d[:,1].min()-1, X_2d[:,1].max()+1, 300))
Z = svm2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
plt.scatter(*X_2d[bc.target==0].T, c='red',  s=12, label='Malignant')
plt.scatter(*X_2d[bc.target==1].T, c='blue', s=12, label='Benign')
sv = svm2d.support_vectors_
plt.scatter(sv[:,0], sv[:,1], s=80, facecolors='none',
            edgecolors='k', linewidths=1.5, label='Support Vectors')
plt.title('SVM Decision Boundary (PCA 2D projection)')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.tight_layout(); plt.show()

---
## 5. Ensemble Methods — Random Forest & XGBoost
**Dataset:** Breast Cancer Wisconsin  
**Task:** Bagging (RF), Boosting (XGBoost), ROC comparison, SHAP explanations


In [ ]:
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed — using GradientBoostingClassifier")

bc = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    bc.data, bc.target, test_size=0.2, random_state=42, stratify=bc.target)

models = {
    'Random Forest':   RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boost':  GradientBoostingClassifier(n_estimators=200, random_state=42),
    'AdaBoost':        AdaBoostClassifier(n_estimators=200, random_state=42),
}
if HAS_XGB:
    models['XGBoost'] = XGBClassifier(n_estimators=200, eval_metric='logloss',
                                       use_label_encoder=False, random_state=42)

plt.figure(figsize=(8, 5))
for name, clf in models.items():
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    auc = roc_auc_score(y_te, proba)
    print(f"{name:20s}  AUC={auc:.4f}  F1={f1_score(y_te, clf.predict(X_te)):.4f}")
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--'); plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves — Ensemble Methods'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)

fi = pd.Series(rf.feature_importances_, index=bc.feature_names).sort_values(ascending=False)
plt.figure(figsize=(10, 4))
fi[:15].plot.bar(color='steelblue')
plt.title('Random Forest — Top 15 Feature Importances')
plt.ylabel('Mean Decrease in Gini'); plt.tight_layout(); plt.show()

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_te)
    shap.summary_plot(shap_values[1], X_te,
                      feature_names=bc.feature_names, show=True)
except ImportError:
    print("pip install shap  to enable SHAP explanations")

---
## 6. Multi-Layer Perceptron (MLP)
**Dataset:** MNIST Digits (8x8, 1797 samples, 10 classes)  
**Task:** Handwritten digit classification with learning curve analysis


In [ ]:
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X = digits.data / 16.0
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=32,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)
mlp.fit(X_tr, y_tr)
print("Test accuracy:", mlp.score(X_te, y_te))
print(classification_report(y_te, mlp.predict(X_te)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mlp.loss_curve_, label='Train loss')
if mlp.validation_scores_ is not None:
    axes[0].plot(mlp.validation_scores_, label='Val score')
axes[0].set(xlabel='Epoch', title='MLP Training Curve'); axes[0].legend()

cm = confusion_matrix(y_te, mlp.predict(X_te))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('MLP Confusion Matrix — Digits')
plt.tight_layout(); plt.show()

In [ ]:
wrong = np.where(mlp.predict(X_te) != y_te)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, idx in zip(axes.ravel(), wrong):
    ax.imshow(X_te[idx].reshape(8,8), cmap='gray_r')
    ax.set_title(f"True={y_te[idx]} Pred={mlp.predict(X_te[idx:idx+1])[0]}")
    ax.axis('off')
plt.suptitle('MLP — Misclassified Digits'); plt.tight_layout(); plt.show()

---
## 7. Convolutional Neural Network (CNN)
**Dataset:** MNIST / CIFAR-10 (via PyTorch/Keras)  
**Task:** Image classification with Conv2D → MaxPool → FC architecture + transfer learning


In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("Install PyTorch:  pip install torch torchvision")

if HAS_TORCH:
    from sklearn.datasets import load_digits
    digits = load_digits()
    X = digits.data.reshape(-1, 1, 8, 8).astype('float32') / 16.0
    y = digits.target.astype('long')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)

    class SmallCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
            self.bn1   = nn.BatchNorm2d(32)
            self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
            self.bn2   = nn.BatchNorm2d(64)
            self.pool  = nn.MaxPool2d(2)
            self.drop  = nn.Dropout(0.3)
            self.fc1   = nn.Linear(64 * 2 * 2, 128)
            self.fc2   = nn.Linear(128, 10)

        def forward(self, x):
            x = F.relu(self.bn1(self.conv1(x)))
            x = self.pool(F.relu(self.bn2(self.conv2(x))))
            x = x.view(x.size(0), -1)
            x = F.relu(self.fc1(self.drop(x)))
            return self.fc2(x)

    model = SmallCNN()
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
                        batch_size=32, shuffle=True)

    history = []
    for epoch in range(30):
        model.train()
        epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            l = loss_fn(model(xb), yb)
            l.backward(); opt.step()
            epoch_loss += l.item()
        history.append(epoch_loss / len(loader))

    model.eval()
    with torch.no_grad():
        logits  = model(torch.tensor(X_te))
        y_pred  = logits.argmax(1).numpy()
    print("CNN Test accuracy:", accuracy_score(y_te, y_pred))
    print(classification_report(y_te, y_pred))

    plt.figure(figsize=(6, 3))
    plt.plot(history); plt.xlabel('Epoch'); plt.ylabel('Train Loss')
    plt.title('CNN Training Loss'); plt.tight_layout(); plt.show()

---
## 8. LSTM — Sequence Classification
**Dataset:** IMDB Sentiment (via custom word2index)  
**Task:** Binary sentiment classification of movie reviews using an LSTM


In [ ]:
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

if HAS_TORCH:
    from sklearn.datasets import fetch_20newsgroups
    from sklearn.feature_extraction.text import CountVectorizer
    import re

    cats = ['rec.sport.hockey', 'sci.med']
    raw_tr = fetch_20newsgroups(subset='train', categories=cats,
                                 remove=('headers','footers','quotes'))
    raw_te = fetch_20newsgroups(subset='test',  categories=cats,
                                 remove=('headers','footers','quotes'))

    def tokenise(text, maxlen=100):
        words = re.sub(r'[^a-z ]', ' ', text.lower()).split()
        return words[:maxlen]

    all_words = [w for doc in raw_tr.data for w in tokenise(doc)]
    vocab = {w: i+2 for i, (w, _) in
             enumerate(pd.Series(all_words).value_counts()[:5000].items())}
    vocab['<pad>'] = 0; vocab['<unk>'] = 1

    def encode(docs, maxlen=100):
        seqs = [[vocab.get(w, 1) for w in tokenise(d)] for d in docs]
        out  = np.zeros((len(seqs), maxlen), dtype='int64')
        for i, s in enumerate(seqs):
            out[i, :len(s)] = s[:maxlen]
        return out

    X_tr_enc = encode(raw_tr.data); X_te_enc = encode(raw_te.data)
    y_tr_t   = torch.tensor(raw_tr.target, dtype=torch.float32)
    y_te_t   = torch.tensor(raw_te.target, dtype=torch.float32)

    class LSTMClassifier(nn.Module):
        def __init__(self, vocab_size=5002, emb_dim=64, hidden=128, n_layers=2):
            super().__init__()
            self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            self.lstm = nn.LSTM(emb_dim, hidden, n_layers,
                                batch_first=True, dropout=0.3)
            self.drop = nn.Dropout(0.4)
            self.fc   = nn.Linear(hidden, 1)

        def forward(self, x):
            e = self.emb(x)
            _, (h, _) = self.lstm(e)
            return self.fc(self.drop(h[-1])).squeeze(1)

    model = LSTMClassifier()
    opt   = torch.optim.Adam(model.parameters(), lr=2e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    X_tr_t  = torch.tensor(X_tr_enc); X_te_t = torch.tensor(X_te_enc)
    loader  = DataLoader(list(zip(X_tr_t, y_tr_t)), batch_size=64, shuffle=True)

    train_losses = []
    for epoch in range(10):
        model.train(); total = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
            total += loss.item()
        train_losses.append(total / len(loader))
        print(f"Epoch {epoch+1:2d}  loss={train_losses[-1]:.4f}")

    model.eval()
    with torch.no_grad():
        logits = model(X_te_t)
        y_pred = (torch.sigmoid(logits) > 0.5).long().numpy()
    print("LSTM Test Accuracy:", accuracy_score(raw_te.target, y_pred))
    print(classification_report(raw_te.target, y_pred,
          target_names=cats))

    plt.figure(figsize=(6, 3))
    plt.plot(train_losses); plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('LSTM Training Loss'); plt.tight_layout(); plt.show()

---
## 9. k-Means Clustering
**Dataset:** Digits (8×8 images) — treated as unlabelled for clustering  
**Task:** Cluster 1797 digit images, evaluate cluster purity, visualise cluster centroids


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)

inertias, silhouettes = [], []
K_range = range(2, 20)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels, sample_size=500))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'o-', color='steelblue')
axes[0].set(xlabel='k', ylabel='WCSS (Inertia)', title='Elbow Method')
axes[1].plot(K_range, silhouettes, 'o-', color='seagreen')
axes[1].axvline(K_range[np.argmax(silhouettes)], color='red', linestyle='--')
axes[1].set(xlabel='k', ylabel='Silhouette Score', title='Silhouette Analysis')
plt.tight_layout(); plt.show()

km10 = KMeans(n_clusters=10, n_init=20, random_state=42)
labels = km10.fit_predict(X)
ari = adjusted_rand_score(digits.target, labels)
sil = silhouette_score(X, labels)
print(f"k=10  ARI={ari:.4f}  Silhouette={sil:.4f}")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(km10.cluster_centers_[i].reshape(8,8), cmap='gray_r')
    ax.set_title(f'Centroid {i}'); ax.axis('off')
plt.suptitle('k-Means Cluster Centroids (k=10)'); plt.tight_layout(); plt.show()

---
## 10. DBSCAN — Density-Based Clustering
**Dataset:** Synthetic 2D datasets — moons and circles (non-convex cluster shapes)  
**Task:** Show DBSCAN succeeds where k-Means fails; noise point detection


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons, make_circles

X_moons, _ = make_moons(n_samples=500, noise=0.08, random_state=42)
X_circles, _ = make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=42)

datasets = [('Moons', X_moons, 0.15, 5),
            ('Circles', X_circles, 0.12, 5)]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (name, X, eps, min_pts) in enumerate(datasets):
    X_s = StandardScaler().fit_transform(X)

    km = KMeans(n_clusters=2, random_state=42)
    db = DBSCAN(eps=eps, min_samples=min_pts)
    km_labels = km.fit_predict(X_s)
    db_labels = db.fit_predict(X_s)

    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise    = (db_labels == -1).sum()

    axes[i][0].scatter(X_s[:,0], X_s[:,1], c=km_labels, cmap='Set1', s=8)
    axes[i][0].set_title(f'{name} — k-Means (k=2)')

    core_mask  = db_labels != -1
    axes[i][1].scatter(X_s[core_mask,0],  X_s[core_mask,1],
                       c=db_labels[core_mask], cmap='Set1', s=8)
    axes[i][1].scatter(X_s[~core_mask,0], X_s[~core_mask,1],
                       c='black', s=15, marker='x', label=f'Noise n={n_noise}')
    axes[i][1].set_title(f'{name} — DBSCAN ({n_clusters} clusters, ε={eps})')
    axes[i][1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
from sklearn.neighbors import NearestNeighbors

X_s = StandardScaler().fit_transform(X_moons)
nbrs = NearestNeighbors(n_neighbors=5).fit(X_s)
dists, _ = nbrs.kneighbors(X_s)
k_dists   = np.sort(dists[:, -1])

plt.figure(figsize=(6, 3))
plt.plot(k_dists); plt.xlabel('Points (sorted)')
plt.ylabel(f'5-NN Distance'); plt.title('k-Distance Graph — Choose ε at elbow')
plt.tight_layout(); plt.show()

---
## 11. Hierarchical Clustering — Agglomerative
**Dataset:** Iris (4 features, 3 species)  
**Task:** Dendrogram construction, Ward vs Complete vs Average linkage comparison


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

iris = load_iris()
X_s = StandardScaler().fit_transform(iris.data)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, method in zip(axes, ['ward', 'complete', 'average']):
    Z = linkage(X_s, method=method)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=20,
               color_threshold=0.7 * max(Z[:,2]))
    ax.set_title(f'Linkage: {method.capitalize()}')
    ax.set_xlabel('Sample index'); ax.set_ylabel('Distance')
plt.suptitle('Hierarchical Clustering Dendrograms — Iris'); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import adjusted_rand_score

linkages = ['ward', 'complete', 'average', 'single']
for lnk in linkages:
    model = AgglomerativeClustering(n_clusters=3, linkage=lnk)
    labels = model.fit_predict(X_s)
    ari = adjusted_rand_score(iris.target, labels)
    print(f"{lnk:10s}  ARI = {ari:.4f}")

---
## 12. Gaussian Mixture Model (GMM) + EM
**Dataset:** Iris (treated as unlabelled)  
**Task:** Soft clustering with BIC model selection, compare to k-Means


In [ ]:
from sklearn.mixture import GaussianMixture

iris = load_iris()
X_s = StandardScaler().fit_transform(iris.data)

bics, aics = [], []
K_range = range(1, 10)
for k in K_range:
    gm = GaussianMixture(n_components=k, n_init=5, random_state=42)
    gm.fit(X_s)
    bics.append(gm.bic(X_s))
    aics.append(gm.aic(X_s))

best_k = K_range[np.argmin(bics)]
print(f"BIC selects K = {best_k}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(K_range, bics, 'o-', label='BIC')
ax.plot(K_range, aics, 's--', label='AIC')
ax.axvline(best_k, color='red', linestyle=':', label=f'Best K={best_k}')
ax.set(xlabel='K', title='BIC / AIC for GMM Model Selection'); ax.legend()
plt.tight_layout(); plt.show()

gm = GaussianMixture(n_components=3, n_init=10, random_state=42)
gm.fit(X_s)
labels = gm.predict(X_s)
probs  = gm.predict_proba(X_s)

ari = adjusted_rand_score(iris.target, labels)
print(f"GMM (K=3) ARI = {ari:.4f}")
print("Cluster weights (π):", gm.weights_.round(3))

pd.DataFrame(probs[:8].round(3),
             columns=[f'P(cluster {i})' for i in range(3)]).head(8)

---
## 13. PCA — Dimensionality Reduction
**Dataset:** Breast Cancer Wisconsin (30 features → 2D)  
**Task:** Explained variance scree plot, 2D visualisation, reconstruction error


In [ ]:
from sklearn.decomposition import PCA

bc = load_breast_cancer()
X_s = StandardScaler().fit_transform(bc.data)

pca_full = PCA().fit(X_s)
ev_ratio = pca_full.explained_variance_ratio_
cum_ev   = np.cumsum(ev_ratio)
n_95 = np.searchsorted(cum_ev, 0.95) + 1
print(f"PCs needed for 95% variance: {n_95}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, 31), ev_ratio, color='steelblue', alpha=0.7)
axes[0].plot(range(1, 31), cum_ev, 'ro-', markersize=4)
axes[0].axhline(0.95, color='k', linestyle='--', label='95%')
axes[0].set(xlabel='PC', ylabel='Variance ratio', title='Scree Plot')
axes[0].legend()

pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_s)
scatter = axes[1].scatter(X_2d[:,0], X_2d[:,1], c=bc.target,
                          cmap='RdBu', s=15, alpha=0.7)
plt.colorbar(scatter, ax=axes[1])
axes[1].set(xlabel='PC1', ylabel='PC2',
            title='Breast Cancer — PCA 2D Projection')
plt.tight_layout(); plt.show()

for k in [2, 5, 10, n_95]:
    pca_k = PCA(n_components=k).fit(X_s)
    X_rec = pca_k.inverse_transform(pca_k.transform(X_s))
    err = np.mean((X_s - X_rec)**2)
    print(f"k={k:2d}  MSE_reconstruction={err:.4f}  "
          f"var_explained={sum(pca_k.explained_variance_ratio_):.3f}")

---
## 14. t-SNE and UMAP — Non-Linear Visualisation
**Dataset:** Digits (8×8, 10 classes)  
**Task:** 2D embedding for visualisation, compare t-SNE vs UMAP


In [ ]:
from sklearn.manifold import TSNE

digits = load_digits()
X_s = StandardScaler().fit_transform(digits.data)

pca50 = PCA(n_components=50).fit_transform(X_s)

tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
            init='pca', random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(pca50)

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_tsne[:,0], X_tsne[:,1], c=digits.target,
                 cmap='tab10', s=6, alpha=0.8)
plt.colorbar(sc, label='Digit Class')
plt.title('t-SNE of MNIST Digits'); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15,
                        min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(pca50)

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(X_umap[:,0], X_umap[:,1], c=digits.target,
                     cmap='tab10', s=6, alpha=0.8)
    plt.colorbar(sc, label='Digit Class')
    plt.title('UMAP of MNIST Digits'); plt.axis('off')
    plt.tight_layout(); plt.show()
except ImportError:
    print("pip install umap-learn  to run UMAP")

---
## 15. Spectral Clustering
**Dataset:** Synthetic non-convex shapes (moons, circles, anisotropic)  
**Task:** Show spectral clustering succeeds on shapes that defeat k-Means


In [ ]:
from sklearn.cluster import SpectralClustering
from sklearn.datasets import make_moons, make_circles

np.random.seed(42)
datasets = {
    'Moons':   make_moons(n_samples=300, noise=0.05)[0],
    'Circles': make_circles(n_samples=300, factor=0.5, noise=0.04)[0],
    'Aniso':   np.dot(np.random.randn(300, 2),
                      [[0.6, -0.6], [-0.4, 0.8]]),
}

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
for row, (name, X) in enumerate(datasets.items()):
    X_s = StandardScaler().fit_transform(X)

    km_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_s)
    sp_labels = SpectralClustering(
        n_clusters=2, affinity='rbf', gamma=2.0,
        assign_labels='kmeans', random_state=42).fit_predict(X_s)

    axes[row][0].scatter(X_s[:,0], X_s[:,1], c=km_labels, cmap='Set1', s=10)
    axes[row][0].set_title(f'{name} — k-Means')

    axes[row][1].scatter(X_s[:,0], X_s[:,1], c=sp_labels, cmap='Set1', s=10)
    axes[row][1].set_title(f'{name} — Spectral Clustering')

plt.suptitle('Spectral vs k-Means on Non-Convex Clusters', y=1.01)
plt.tight_layout(); plt.show()

---
## 16. Association Rule Mining — Apriori & FP-Growth
**Dataset:** Online Retail II UCI dataset (simulated basket data if unavailable)  
**Task:** Mine frequent itemsets and association rules; filter by lift and conviction


In [ ]:
try:
    from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
    HAS_MLXTEND = True
except ImportError:
    HAS_MLXTEND = False
    print("pip install mlxtend")

if HAS_MLXTEND:
    np.random.seed(42)
    items = ['bread','milk','butter','eggs','jam','cheese','yoghurt','juice']
    n_transactions = 2000
    transactions = []
    probs = [0.5, 0.45, 0.35, 0.4, 0.25, 0.3, 0.28, 0.32]
    co_occur = {'bread': ['butter','jam'], 'milk': ['yoghurt','cheese'],
                'eggs':  ['butter','cheese']}
    for _ in range(n_transactions):
        basket = set(i for i, p in zip(items, probs) if np.random.rand() < p)
        for trigger, companions in co_occur.items():
            if trigger in basket:
                for c in companions:
                    if np.random.rand() < 0.6:
                        basket.add(c)
        if len(basket) >= 2:
            transactions.append(basket)

    basket_df = pd.DataFrame([{i: (i in t) for i in items} for t in transactions])

    import time
    t0 = time.time()
    fi_apriori = apriori(basket_df, min_support=0.05, use_colnames=True)
    t_apr = time.time() - t0

    t0 = time.time()
    fi_fp = fpgrowth(basket_df, min_support=0.05, use_colnames=True)
    t_fp = time.time() - t0

    print(f"Apriori  — {len(fi_apriori)} frequent itemsets  ({t_apr:.4f}s)")
    print(f"FP-Growth — {len(fi_fp)}  frequent itemsets  ({t_fp:.4f}s)")

    rules = association_rules(fi_fp, metric='lift', min_threshold=1.2)
    rules['conviction'] = ((1 - rules['consequent support']) /
                           (1 - rules['confidence'] + 1e-10))
    rules = rules.sort_values('lift', ascending=False)

    display_cols = ['antecedents','consequents','support','confidence','lift','conviction']
    print(rules[display_cols].head(10).to_string(index=False))

In [ ]:
if HAS_MLXTEND and len(rules) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].scatter(rules['support'], rules['confidence'],
                    c=rules['lift'], cmap='YlOrRd', s=40, alpha=0.7)
    plt.colorbar(axes[0].collections[0], ax=axes[0], label='Lift')
    axes[0].set(xlabel='Support', ylabel='Confidence',
                title='Support vs Confidence (colour=Lift)')

    top = rules.nlargest(10, 'lift')
    labels = [f"{list(r.antecedents)}→{list(r.consequents)}"
              for _, r in top.iterrows()]
    axes[1].barh(range(len(top)), top['lift'].values, color='steelblue')
    axes[1].set_yticks(range(len(top)))
    axes[1].set_yticklabels(labels, fontsize=8)
    axes[1].set(xlabel='Lift', title='Top 10 Rules by Lift')
    plt.tight_layout(); plt.show()

---
## 17. Model Evaluation — ROC, PR Curves, Cross-Validation
**Dataset:** Breast Cancer Wisconsin  
**Task:** Compare 5 classifiers — ROC, PR curves, learning curves, McNemar's test


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve
from statsmodels.stats.contingency_tables import mcnemar

bc = load_breast_cancer()
X_s = StandardScaler().fit_transform(bc.data)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_s, bc.target, test_size=0.2, random_state=42, stratify=bc.target)

classifiers = {
    'Logistic Reg': LogisticRegression(max_iter=1000),
    'k-NN (k=7)':   KNeighborsClassifier(n_neighbors=7),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':     SVC(kernel='rbf', C=10, gamma='scale', probability=True),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results = {}
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    proba   = clf.predict_proba(X_te)[:, 1]
    y_pred  = clf.predict(X_te)
    fpr, tpr, _ = roc_curve(y_te, proba)
    pre, rec, _ = precision_recall_curve(y_te, proba)
    auc   = roc_auc_score(y_te, proba)
    ap    = average_precision_score(y_te, proba)
    results[name] = {'y_pred': y_pred, 'auc': auc, 'ap': ap,
                     'f1': f1_score(y_te, y_pred)}
    axes[0].plot(fpr, tpr, label=f'{name} ({auc:.3f})')
    axes[1].plot(rec, pre, label=f'{name} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--'); axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curves')
axes[0].legend(fontsize=8)
prevalence = y_te.mean()
axes[1].axhline(prevalence, color='k', linestyle='--', label=f'Baseline={prevalence:.3f}')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'Model':20s}  {'AUC-ROC':8s}  {'AUC-PR':8s}  {'F1':8s}")
for name, r in results.items():
    print(f"{name:20s}  {r['auc']:.4f}    {r['ap']:.4f}    {r['f1']:.4f}")

In [ ]:
rf_pred  = results['Random Forest']['y_pred']
svm_pred = results['SVM (RBF)']['y_pred']

b = np.sum((rf_pred == y_te) & (svm_pred != y_te))
c = np.sum((rf_pred != y_te) & (svm_pred == y_te))
contingency = np.array([[np.sum((rf_pred==y_te)&(svm_pred==y_te)),b],
                         [c, np.sum((rf_pred!=y_te)&(svm_pred!=y_te))]])
result = mcnemar(contingency, exact=False, correction=True)
print(f"McNemar Test — RF vs SVM")
print(f"b={b}, c={c}")
print(f"χ² = {result.statistic:.4f}  p-value = {result.pvalue:.4f}")
print("Significant difference" if result.pvalue < 0.05 else "No significant difference")

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
train_sizes, train_sc, val_sc = learning_curve(
    clf, X_s, bc.target, cv=5, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1)

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, train_sc.mean(1), 'o-', label='Train F1')
plt.fill_between(train_sizes,
                 train_sc.mean(1)-train_sc.std(1),
                 train_sc.mean(1)+train_sc.std(1), alpha=0.2)
plt.plot(train_sizes, val_sc.mean(1), 's-', label='CV F1')
plt.fill_between(train_sizes,
                 val_sc.mean(1)-val_sc.std(1),
                 val_sc.mean(1)+val_sc.std(1), alpha=0.2)
plt.xlabel('Training set size'); plt.ylabel('F1 Score')
plt.title('Random Forest Learning Curve'); plt.legend(); plt.tight_layout(); plt.show()

---
## 18. Feature Engineering & Selection
**Dataset:** California Housing (20,640 samples, 8 features)  
**Task:** Missing value imputation, feature selection (Filter + Wrapper + Embedded), SHAP


In [ ]:
from sklearn.feature_selection import (
    SelectKBest, mutual_info_regression, RFECV)
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SKPipe

housing = fetch_california_housing()
X, y = housing.data.copy(), housing.target

np.random.seed(42)
mask = np.random.rand(*X.shape) < 0.07
X[mask] = np.nan
print(f"Missing values injected: {mask.sum()}")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

pipe_full = SKPipe([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('model',  RandomForestRegressor(n_estimators=100, random_state=42))
])
pipe_full.fit(X_tr, y_tr)
from sklearn.metrics import mean_squared_error
rmse = mean_squared_error(y_te, pipe_full.predict(X_te), squared=False)
print(f"RMSE (all features): {rmse:.4f}")

X_imp = SimpleImputer(strategy='median').fit_transform(X_tr)
X_s   = StandardScaler().fit_transform(X_imp)

mi_scores = mutual_info_regression(X_s, y_tr, random_state=42)
mi_df = pd.Series(mi_scores, index=housing.feature_names).sort_values(ascending=False)
print("Mutual Information scores:")
print(mi_df.round(4))

plt.figure(figsize=(7, 3))
mi_df.plot.bar(color='seagreen')
plt.title('Mutual Information — California Housing Features')
plt.ylabel('MI Score'); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.feature_selection import RFECV

rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
X_tr_imp = SimpleImputer(strategy='median').fit_transform(X_tr)
X_te_imp  = SimpleImputer(strategy='median').fit_transform(X_te)

selector = RFECV(rf_base, step=1, cv=3, scoring='neg_root_mean_squared_error',
                 n_jobs=-1)
selector.fit(X_tr_imp, y_tr)
print(f"Optimal features: {selector.n_features_} / {X_tr.shape[1]}")
print("Selected:", [housing.feature_names[i]
                    for i in np.where(selector.support_)[0]])

plt.figure(figsize=(6, 3))
plt.plot(range(1, len(selector.cv_results_['mean_test_score'])+1),
         -selector.cv_results_['mean_test_score'], 'o-')
plt.xlabel('Number of features'); plt.ylabel('CV RMSE')
plt.title('RFE Cross-Validated RMSE'); plt.tight_layout(); plt.show()

---
## 19. Handling Imbalanced Data — SMOTE & Threshold Tuning
**Dataset:** Synthetic imbalanced binary dataset (5000 samples, 5% minority)  
**Task:** Compare raw, under-sampled, SMOTE, class-weighted, threshold-tuned classifiers


In [ ]:
X_imb, y_imb = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42)

print(f"Class distribution: {np.bincount(y_imb)}")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)
X_tr_s = StandardScaler().fit_transform(X_tr)
X_te_s  = StandardScaler().fit_transform(X_te)

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBL = True
except ImportError:
    HAS_IMBL = False
    print("pip install imbalanced-learn")

results_imb = {}

clf_raw = RandomForestClassifier(n_estimators=200, random_state=42)
clf_raw.fit(X_tr_s, y_tr)
results_imb['No resampling'] = clf_raw.predict(X_te_s)

clf_wt = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
clf_wt.fit(X_tr_s, y_tr)
results_imb['Class weights'] = clf_wt.predict(X_te_s)

if HAS_IMBL:
    rus = RandomUnderSampler(random_state=42)
    X_us, y_us = rus.fit_resample(X_tr_s, y_tr)
    clf_us = RandomForestClassifier(n_estimators=200, random_state=42)
    clf_us.fit(X_us, y_us)
    results_imb['Under-sampling'] = clf_us.predict(X_te_s)

    smt = SMOTE(random_state=42)
    X_sm, y_sm = smt.fit_resample(X_tr_s, y_tr)
    clf_sm = RandomForestClassifier(n_estimators=200, random_state=42)
    clf_sm.fit(X_sm, y_sm)
    results_imb['SMOTE'] = clf_sm.predict(X_te_s)

proba_wt = clf_wt.predict_proba(X_te_s)[:, 1]
pre, rec, thr = precision_recall_curve(y_te, proba_wt)
f1s = 2*pre*rec/(pre+rec+1e-9)
best_thr = thr[np.argmax(f1s[:-1])]
results_imb['Threshold tuning'] = (proba_wt >= best_thr).astype(int)
print(f"Optimal threshold: {best_thr:.3f}")

print(f"
{'Method':20s}  {'Recall':7s}  {'Precision':9s}  {'F1':6s}")
from sklearn.metrics import recall_score, precision_score
for name, preds in results_imb.items():
    rec  = recall_score(y_te, preds)
    prec = precision_score(y_te, preds, zero_division=0)
    f1   = f1_score(y_te, preds)
    print(f"{name:20s}  {rec:.4f}   {prec:.4f}      {f1:.4f}")

---
## 20. Text Mining — TF-IDF Classification & LDA Topic Modelling
**Dataset:** 20 Newsgroups (5 categories)  
**Task:** TF-IDF text classification + LDA topic discovery


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

cats = ['rec.sport.hockey', 'sci.med', 'comp.graphics',
        'talk.politics.guns', 'rec.autos']
news = fetch_20newsgroups(subset='all', categories=cats,
                          remove=('headers','footers','quotes'))

cv = CountVectorizer(max_df=0.95, min_df=5, max_features=5000,
                     stop_words='english')
dtm = cv.fit_transform(news.data)
vocab = np.array(cv.get_feature_names_out())

lda = LatentDirichletAllocation(n_components=5, max_iter=10,
                                 learning_method='online', random_state=42)
lda.fit(dtm)

print("Top 10 words per topic:")
for t_idx, topic in enumerate(lda.components_):
    top_words = vocab[topic.argsort()[:-11:-1]]
    print(f"  Topic {t_idx+1}: {', '.join(top_words)}")

In [ ]:
tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, sublinear_tf=True,
                              ngram_range=(1,2), stop_words='english')),
    ('clf',   LogisticRegression(max_iter=500, C=5))
])

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    news.data, news.target, test_size=0.2, random_state=42, stratify=news.target)
tfidf_pipe.fit(X_tr_t, y_tr_t)
print("Logistic + TF-IDF accuracy:", tfidf_pipe.score(X_te_t, y_te_t))
print(classification_report(y_te_t, tfidf_pipe.predict(X_te_t),
      target_names=[c.split('.')[-1] for c in cats]))

---
## 21. End-to-End Production Pipeline
**Dataset:** California Housing (regression) and Breast Cancer (classification)  
**Task:** ColumnTransformer + Pipeline + cross-validate + Bayesian HPO + SHAP


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

bc = load_breast_cancer()
X_df = pd.DataFrame(bc.data, columns=bc.feature_names)
y    = bc.target

num_cols = bc.feature_names.tolist()

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])
preprocessor = ColumnTransformer([('num', num_pipe, num_cols)])

full_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf',  RandomForestClassifier(n_estimators=200,
                                    class_weight='balanced',
                                    random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.model_selection import cross_validate
cv_res = cross_validate(full_pipe, X_df, y, cv=cv,
    scoring=['f1_weighted','roc_auc','recall_weighted'],
    return_train_score=True, n_jobs=-1)

print("5-Fold Cross-Validation Results:")
for metric in ['f1_weighted','roc_auc','recall_weighted']:
    tr = cv_res[f'train_{metric}']
    te = cv_res[f'test_{metric}']
    print(f"  {metric:20s}  train {tr.mean():.4f}±{tr.std():.4f}  "
          f"test {te.mean():.4f}±{te.std():.4f}")

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    X_tr, X_te, y_tr, y_te = train_test_split(
        bc.data, bc.target, test_size=0.2, random_state=42, stratify=bc.target)
    X_tr_s = StandardScaler().fit_transform(X_tr)
    X_te_s  = StandardScaler().fit_transform(X_te)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth':    trial.suggest_int('max_depth', 3, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features': trial.suggest_categorical('max_features',
                                                       ['sqrt', 'log2', None]),
        }
        clf = RandomForestClassifier(**params, class_weight='balanced',
                                     random_state=42, n_jobs=-1)
        return cross_val_score(clf, X_tr_s, y_tr, cv=3, scoring='roc_auc').mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=40, show_progress_bar=False)

    print("Best params:", study.best_params)
    print(f"Best AUC-ROC: {study.best_value:.4f}")

    best_clf = RandomForestClassifier(**study.best_params,
                                      class_weight='balanced', random_state=42)
    best_clf.fit(X_tr_s, y_tr)
    print("Test AUC-ROC:", roc_auc_score(y_te, best_clf.predict_proba(X_te_s)[:,1]))
except ImportError:
    print("pip install optuna  for Bayesian hyperparameter optimisation")